## Notebook for PVS Geometric Analysis

This notebook loads the data generated by `compute-selberg` and performs the initial geometric analysis as described in the project's research plan. Our goal is to test the core hypothesis: that geometric invariants can distinguish between cases where `p+2` is prime and where `p+2` is composite.

### Step 1: Configuration and Setup

First, we import the necessary libraries and define our input file. Make sure `small.parquet` (or a larger file you generate) is in the same directory as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sympy import primepi, isprime, factorint

# --- Configuration ---
DATA_FILE = 'small.parquet' # The output from 'poetry run compute-selberg'
MAX_PRIMES_FOR_PVS = 100 # We'll use the first 100 primes for our vector space basis

sns.set_theme(style="whitegrid")

### Step 2: Load the Sieve Weights

We load the `n` and `W` (squared Selberg weight) columns from the Parquet file.

In [ ]:
try:
    df = pd.read_parquet(DATA_FILE)
    print(f"Successfully loaded {len(df)} records from {DATA_FILE}")
    print("Data preview:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: Data file '{DATA_FILE}' not found.")
    print("Please run 'poetry run compute-selberg --x <number> --outfile {DATA_FILE}' first.")

### Step 3: Define PVS and Geometric Functions

Here, we'll implement the core functions to create the geometric representation of our numbers as described in the research architecture.

In [ ]:
def get_pvs_vector(n, prime_map):
    """Converts an integer n to its Prime Vector Space vector."""
    factors = factorint(n)
    vec = np.zeros(len(prime_map), dtype=np.int32)
    for p, exponent in factors.items():
        if p in prime_map:
            vec[prime_map[p]] = exponent
    return vec

def compute_displacement_vector(v_n, v_n_plus_2):
    """Calculates the displacement vector δ(n) = ν(n+2) - ν(n)."""
    return v_n_plus_2 - v_n

def l1_norm(vector):
    """Calculates the L1 norm of a vector, which is Ω(n), the sum of exponents."""
    return np.sum(np.abs(vector))

### Step 4: Process the Data

Now we'll iterate through our dataframe to identify primes `p` and calculate the geometric invariants for the pairs `(p, p+2)`.

In [ ]:
# Generate the first MAX_PRIMES_FOR_PVS primes and a map for quick lookups
from sympy import primerange
primes_list = list(primerange(0, 500)) # Adjust upper bound based on MAX_PRIMES_FOR_PVS
prime_map = {p: i for i, p in enumerate(primes_list)}

results = []
# We look at n and n+2, so we iterate up to len(df) - 2
for i in range(len(df) - 2):
    n = df.loc[i, 'n']
    n_plus_2 = df.loc[i+2, 'n']
    
    # Check if n is a prime
    if isprime(n):
        p = n
        
        # Get PVS vectors
        v_p = get_pvs_vector(p, prime_map)
        v_p_plus_2 = get_pvs_vector(p + 2, prime_map)
        
        # Calculate displacement vector and its norm
        delta_vec = compute_displacement_vector(v_p, v_p_plus_2)
        delta_norm = l1_norm(delta_vec)
        
        # Classify the outcome
        is_twin = isprime(p + 2)
        
        results.append({
            'p': p,
            'p_plus_2_is_prime': is_twin,
            'delta_l1_norm': delta_norm
        })

analysis_df = pd.DataFrame(results)
print(f"Processed {len(analysis_df)} primes.")
print("Analysis data preview:")
display(analysis_df.head())

### Step 5: Visualize the Results

This is the moment of truth. We will create histograms to see if the distribution of the L1 norm of the displacement vector (`delta_l1_norm`) is different for twin primes versus composites. This directly tests your 'Displacement Vector Hypothesis'.

In [ ]:
plt.figure(figsize=(14, 7))
sns.histplot(data=analysis_df, x='delta_l1_norm', hue='p_plus_2_is_prime', 
             multiple='dodge', shrink=0.8, bins=np.arange(0, 20) - 0.5, stat='probability')

plt.title('Distribution of Displacement Vector L1 Norm (δ(p))', fontsize=16)
plt.xlabel('L1 Norm of δ(p) = ν(p+2) - ν(p)', fontsize=12)
plt.ylabel('Proportion', fontsize=12)
plt.xticks(np.arange(0, 20))
plt.legend(title='p+2 is Prime', labels=['False', 'True'])
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

# --- Interpretation ---
print("Hypothesis Check:")
print("If the 'Displacement Vector Hypothesis' is correct, we expect to see a clear separation between the blue (p+2 is composite) and orange (p+2 is prime) bars.")
print("For a true twin prime (p, p+2), the displacement vector is e_{p+2} - e_p, so its L1 norm should be 2. We should see a strong spike for the orange bars at x=2.")